## Phase 1: Data Set Details
**Goal:** understand the data before any analysis or modelling.

# 1.Setup and Load

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import os

DATA_PATH = '../data/stock_risk_prediction_dataset.xlsx'
CLEAN_PATH = '../data/cleaned_stock_risk_prediction_dataset.xlsx'

#By default, pandas hides middle columns of wide tables and shows ... instead. None means "no limit", so all 11 columns always show when you print a table.
pd.set_option('display.max_columns', None)

# Seperating the coulumns based on their data types
NUMERIC = ["current_stock", "avg_daily_demand", "lead_time_days", "reorder_point",
           "sales_last_30_days", "stock_turnover_ratio", "forecast_error"]
CATEGORICAL = ["season", "item_category", "supplier_reliability"]
TARGET = "stock_status"


df = pd.read_excel(DATA_PATH)
df.head()


## 2. Shape , Columns and data Types

In [ ]:
print("Rows x Columns:", df.shape, "  (expected 5000 x 11)")

expected_cols = NUMERIC + CATEGORICAL + [TARGET]
print("Missing expected columns:", [c for c in expected_cols if c not in df.columns] or "none")
print("Unexpected extra columns:", [c for c in df.columns if c not in expected_cols] or "none")

In [ ]:
df.info()

**Check:** numeric columns should show `int64`/`float64`, categorical and target columns `object`. A number stored as `object` means there is text in that column that must be fixed.


In [41]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2),
})
if missing.missing_count.sum():
    display(missing[missing.missing_count > 0])
else:
    print("No missing values")

No missing values


In [42]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [ ]:
EXPECTED_VALUES = {
    "season": {"peak", "off-peak"},
    "item_category": {"essential", "non-essential"},
    "supplier_reliability": {"high", "low"},
    "stock_status": {"stockout", "overstock"},
}

for col, expected in EXPECTED_VALUES.items():
    found = set(df[col].dropna().unique())
    print(f"{col:22s} found: {sorted(map(str, found))}")
    unexpected = found - expected
    if unexpected:
        print(f"   -> unexpected labels (case/spacing issues?): {sorted(map(str, unexpected))}")

season                 found: ['off-peak', 'peak']
item_category          found: ['essential', 'non-essential']
supplier_reliability   found: ['high', 'low']
stock_status           found: ['overstock', 'stockout']


In [ ]:
rows = []
for col in NUMERIC:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = int(((df[col] < low) | (df[col] > high)).sum())
    rows.append({"column": col, "lower_fence": round(low, 2), "upper_fence": round(high, 2),
                 "outlier_count": n, "outlier_percent": round(n / len(df) * 100, 2)})
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), NUMERIC):
    ax.boxplot(df[col].dropna())
    ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
df[NUMERIC].describe().T.round(2)

In [ ]:
for col in CATEGORICAL:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

In [ ]:
counts = df[TARGET].value_counts(dropna=False)
percent = df[TARGET].value_counts(normalize=True, dropna=False).mul(100).round(1)
display(pd.DataFrame({"count": counts, "percent": percent}))

ax = counts.plot(kind="bar", color=["#d9534f", "#5bc0de"], figsize=(5, 3.5), rot=0)
ax.set_title("Target distribution: stock_status")
ax.set_ylabel("rows")
plt.tight_layout()
plt.show()

if percent.min() < 30:
    print("Minority class is under 30%: plan to use class_weight='balanced' or SMOTE in model building.")
else:
    print("Classes are reasonably balanced.")

In [ ]:
clean = df.copy()

# 1. standardise text labels
for col in CATEGORICAL + [TARGET]:
    clean[col] = clean[col].astype("string").str.strip().str.lower()

# 2. remove duplicates
before = len(clean)
clean = clean.drop_duplicates()
print("Duplicates removed:", before - len(clean))

# 3. drop rows with missing target (cannot train on them)
before = len(clean)
clean = clean.dropna(subset=[TARGET])
print("Rows dropped (missing target):", before - len(clean))

# 4. fill missing feature values: median for numeric, mode for categorical
for col in NUMERIC:
    if clean[col].isnull().any():
        clean[col] = clean[col].fillna(clean[col].median())
for col in CATEGORICAL:
    if clean[col].isnull().any():
        clean[col] = clean[col].fillna(clean[col].mode()[0])

# 5. drop impossible rows
bad = (clean["current_stock"] < 0) | (clean["avg_daily_demand"] < 0) | (clean["lead_time_days"] < 0)
print("Rows dropped (negative stock/demand/lead time):", int(bad.sum()))
clean = clean[~bad]

print("\nFinal clean shape:", clean.shape)
print("Missing values left:", int(clean.isnull().sum().sum()))

In [ ]:
os.makedirs(os.path.dirname(CLEAN_PATH), exist_ok=True)
clean.to_csv(CLEAN_PATH, index=False)
print("Saved ->", CLEAN_PATH)
clean.head()